# Classifier Head Training on Top of Milvus Embeddings

This notebook mirrors `exploitation_zone/classifier_training.py` and walks through
the full pipeline interactively:

1. **Setup** — connect to MinIO and Milvus.
2. **Labels** — read the `category` column from the exploitation-zone metadata.
3. **Embeddings** — fetch the matching vectors from each Milvus collection.
4. **Training** — fit a logistic-regression head per modality with stratified CV.
5. **Persistence** — save the fitted model + metrics report to MinIO.

**Why a head?** PANNs CNN14 and CLIP ViT-B/32 are frozen pretrained
feature extractors. The most cost-effective way to add a classifier on a small to
medium dataset is to train a thin linear head on top of those embeddings — it
trains in seconds on CPU, generalises well thanks to the pretrained representation,
and avoids overfitting that a from-scratch deep model would suffer.

**Why no text classifier?** The text description embedded into
`sound_text_embeddings` starts with `"This is a {category} sound."`, so the
embedding already encodes the label. Training a classifier on it would
produce a leaky, meaningless score. Text embeddings are still used for
semantic search but are intentionally excluded from supervised training.

| Modality | Collection | Dim | Head |
|---|---|---|---|
| audio    | `sound_audio_embeddings`    | 2048 | Logistic regression (balanced) |
| cymatics | `sound_cymatics_embeddings` | 512  | Logistic regression (balanced) |

## 1. Setup & Connections

In [ ]:
import os, sys

# Project root so we can import shared helpers and the exploitation-zone modules.
_NB_DIR  = os.path.abspath("")
_EZ_DIR  = os.path.abspath(os.path.join(_NB_DIR, ".."))
_PROJECT = os.path.abspath(os.path.join(_EZ_DIR, ".."))
for p in (_PROJECT, _EZ_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(os.path.join(_PROJECT, ".env"))

import io, json
import numpy as np
import pandas as pd

In [ ]:
# ---- MinIO ----
from shared.minio_helpers import create_minio_client

minio_client = create_minio_client()
print("MinIO connected:", [b.name for b in minio_client.list_buckets()])

In [ ]:
# ---- Milvus ----
from milvus_embeddings import connect_milvus, AUDIO_COLLECTION, CYMATICS_COLLECTION

milvus_client = connect_milvus()
print("Existing collections:", milvus_client.list_collections())

## 2. Load Labels from the Exploitation-Zone Metadata

We use the `category` column as the supervision signal. Rows with missing,
empty or placeholder categories (`"-"`, `nan`, `None`) are dropped, and we
then prune any category that has fewer than `MIN_SAMPLES_PER_CLASS` samples
so that the stratified split is well-defined.

In [ ]:
from milvus_embeddings import load_exploitation_metadata
from classifier_training import (
    collect_labelled_uuids,
    class_distribution,
    trim_to_min_class_size,
    MIN_SAMPLES_PER_CLASS,
    MIN_CLASSES,
)

rows = load_exploitation_metadata(minio_client)
raw_labels = collect_labelled_uuids(rows)
labels     = trim_to_min_class_size(raw_labels, MIN_SAMPLES_PER_CLASS)
dist       = class_distribution(labels)

print(f"Labelled rows: {len(raw_labels)} → {len(labels)} after pruning classes "
      f"with < {MIN_SAMPLES_PER_CLASS} samples.")
print(f"\nClass distribution after pruning ({len(dist)} classes):")
for cat, n in sorted(dist.items(), key=lambda kv: -kv[1]):
    print(f"  {cat:.<40s} {n}")

## 3. Fetch Embeddings from Milvus

For each modality we query the corresponding Milvus collection with a `uuid in [...]`
filter to retrieve the vectors of the labelled rows. Missing UUIDs (e.g. a row whose
audio failed to embed) are silently dropped during the alignment step.

In [ ]:
from classifier_training import MODALITIES, fetch_embeddings, build_xy

uuids = list(labels.keys())
datasets = {}
for m in MODALITIES:
    vectors = fetch_embeddings(milvus_client, m["collection"], m["vector_field"], uuids)
    X, y, used = build_xy(labels, vectors)
    datasets[m["key"]] = {"X": X, "y": y, "uuids": used, "modality": m}
    print(f"  {m['key']:.<10s} X={X.shape}, classes={len(set(y))}")

## 4. Train a Logistic-Regression Head per Modality

The head is a `StandardScaler → LogisticRegression(class_weight="balanced")`
pipeline. We hold out 20\% of the data as a test set, run 5-fold stratified
cross-validation on the remaining 80\% for the generalisation estimate, refit
on the full training portion and evaluate once on the held-out test set.

In [ ]:
from classifier_training import train_and_evaluate, class_distribution_arr

trained = {}
for key, ds in datasets.items():
    if ds["X"].size == 0:
        print(f"  {key}: no embeddings — skipping.")
        continue
    usable = {c for c, n in class_distribution_arr(ds["y"]).items() if n >= MIN_SAMPLES_PER_CLASS}
    if len(usable) < MIN_CLASSES:
        print(f"  {key}: only {len(usable)} usable class(es) — skipping.")
        continue
    mask = np.asarray([cls in usable for cls in ds["y"]])
    X, y = ds["X"][mask], ds["y"][mask]

    model, metrics = train_and_evaluate(X, y)
    metrics["modality"]   = key
    metrics["collection"] = ds["modality"]["collection"]
    trained[key] = {"model": model, "metrics": metrics}
    print(
        f"  {key:.<10s} CV acc {metrics['cv_accuracy_mean']:.3f} ± {metrics['cv_accuracy_std']:.3f} "
        f"| test acc {metrics['test_accuracy']:.3f} "
        f"| test F1 (macro) {metrics['test_f1_macro']:.3f}"
    )

### 4.1 Summary table

In [ ]:
summary = pd.DataFrame(
    [
        {
            "modality":       k,
            "n_samples":      m["metrics"]["n_samples"],
            "n_classes":      m["metrics"]["n_classes"],
            "cv_acc_mean":    m["metrics"]["cv_accuracy_mean"],
            "cv_acc_std":     m["metrics"]["cv_accuracy_std"],
            "cv_f1_macro":    m["metrics"]["cv_f1_macro_mean"],
            "test_acc":       m["metrics"]["test_accuracy"],
            "test_f1_macro":  m["metrics"]["test_f1_macro"],
        }
        for k, m in trained.items()
    ]
)
summary

## 5. Persist Models and Reports to MinIO

Each fitted pipeline is pickled with `joblib` and the metrics dictionary is
stored alongside it as JSON under `exploitation-zone/models/`. The consumption
layer can later load these artefacts to serve calibrated predictions for the
audio and cymatics classification tasks.

In [ ]:
from classifier_training import save_model_to_minio

modalities_by_key = {m["key"]: m for m in MODALITIES}
for key, item in trained.items():
    modality = modalities_by_key[key]
    model_key, metrics_key = save_model_to_minio(
        minio_client, modality, item["model"], item["metrics"],
    )
    print(f"  {key:.<10s} saved model:  {model_key}")
    print(f"            saved report: {metrics_key}")

## Summary

We have:

1. Read the `category` labels from the exploitation-zone metadata and pruned
   under-represented classes.
2. Fetched the three modality embeddings from Milvus and aligned them with the
   labelled UUIDs.
3. Trained a logistic-regression head per modality with stratified 5-fold CV.
4. Persisted the fitted models and the metrics reports to MinIO.

**Next step** — open `classifier_validation.ipynb` to dive deeper into the
generalisation metrics (per-class precision and recall, confusion matrices,
learning curves) and compare the three modalities side by side.